In [ ]:
!pip install -q datasets==2.19.0

from datasets import load_dataset

ds = load_dataset("Jeneral/fer-2013", trust_remote_code=True)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['img_bytes', 'labels'],
        num_rows: 28709
    })
    test: Dataset({
        features: ['img_bytes', 'labels'],
        num_rows: 7178
    })
})


In [ ]:
from collections import Counter

label_names = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
counts = Counter(ds["train"]["labels"])
for i, name in enumerate(label_names):
    print(name, counts[i])

angry 3995
disgust 436
fear 4097
happy 7215
neutral 4965
sad 4830
surprise 3171


In [ ]:
import io
import numpy as np
from PIL import Image
import torch

def decode_image(example):
    img = Image.open(io.BytesIO(example["img_bytes"])).convert("L").resize((48, 48))
    example["pixels"] = np.array(img, dtype=np.float32) / 255.0
    return example

ds = ds.map(decode_image)

X_train = torch.tensor(np.stack(ds["train"]["pixels"]), dtype=torch.float32).unsqueeze(1)
y_train = torch.tensor(ds["train"]["labels"], dtype=torch.long)

X_test = torch.tensor(np.stack(ds["test"]["pixels"]), dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(ds["test"]["labels"], dtype=torch.long)

print(X_train.shape, X_test.shape)

torch.Size([28709, 1, 48, 48]) torch.Size([7178, 1, 48, 48])


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train.numpy()),
    y=y_train.numpy()
)
class_weights_t = torch.tensor(class_weights, dtype=torch.float)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

device = "cuda" if torch.cuda.is_available() else "cpu"
cnn_model = SimpleCNN().to(device)
print(cnn_model)
print(device)

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=4608, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=7, bias=True)
)
cuda


In [ ]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss(weight=class_weights_t.to(device))
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

EPOCHS = 15
for epoch in range(EPOCHS):
    cnn_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = cnn_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss / len(train_loader))

1 1.875620803217049
2 1.6930226655739187
3 1.564384724886752
4 1.457330301238062
5 1.3604694547791258
6 1.2568569799308522
7 1.1857927343362158
8 1.111390799483106
9 1.0439192626152378
10 0.9927322200783643
11 0.9486445577479152
12 0.887703993936424
13 0.837630175668041
14 0.7868036919155206
15 0.7456789973185695


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

cnn_model.eval()
with torch.no_grad():
    test_outputs = cnn_model(X_test.to(device))
    preds = torch.argmax(test_outputs, dim=1).cpu().numpy()

cnn_accuracy = accuracy_score(y_test.numpy(), preds)
print(cnn_accuracy)
print(classification_report(y_test.numpy(), preds, target_names=label_names))

0.5433268319866258
              precision    recall  f1-score   support

       angry       0.42      0.50      0.46       958
     disgust       0.41      0.58      0.48       111
        fear       0.41      0.37      0.39      1024
       happy       0.78      0.71      0.74      1774
     neutral       0.49      0.54      0.51      1233
         sad       0.42      0.36      0.38      1247
    surprise       0.66      0.75      0.70       831

    accuracy                           0.54      7178
   macro avg       0.51      0.54      0.52      7178
weighted avg       0.55      0.54      0.54      7178



In [ ]:
torch.save(cnn_model.state_dict(), "cnn_model.pt")

import pickle
with open("cnn_label_names.pkl", "wb") as f:
    pickle.dump(label_names, f)

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

class ResNetFERDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]
        img = F.interpolate(img.unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False).squeeze(0)
        img = img.repeat(3, 1, 1)
        img = (img - self.mean) / self.std
        return img, self.y[idx]

resnet_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet_model.fc = nn.Linear(resnet_model.fc.in_features, 7)
resnet_model = resnet_model.to(device)
print(resnet_model.fc)

Linear(in_features=512, out_features=7, bias=True)


In [ ]:
train_loader_resnet = DataLoader(ResNetFERDataset(X_train, y_train), batch_size=64, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss(weight=class_weights_t.to(device))
optimizer = optim.Adam(resnet_model.parameters(), lr=0.0001)

EPOCHS = 6
for epoch in range(EPOCHS):
    resnet_model.train()
    total_loss = 0
    for xb, yb in train_loader_resnet:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = resnet_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss / len(train_loader_resnet))

1 1.2675829232402793
2 0.8342829673221223
3 0.5498656774217674
4 0.2847751836999753
5 0.127879019386917
6 0.07016404024543103


In [ ]:
test_loader_resnet = DataLoader(ResNetFERDataset(X_test, y_test), batch_size=64, shuffle=False)

resnet_model.eval()
all_preds = []
with torch.no_grad():
    for xb, yb in test_loader_resnet:
        xb = xb.to(device)
        out = resnet_model(xb)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_preds.extend(preds)

resnet_accuracy = accuracy_score(y_test.numpy(), all_preds)
print(resnet_accuracy)
print(classification_report(y_test.numpy(), all_preds, target_names=label_names))

0.6490665923655614
              precision    recall  f1-score   support

       angry       0.66      0.47      0.55       958
     disgust       0.82      0.59      0.68       111
        fear       0.50      0.50      0.50      1024
       happy       0.87      0.83      0.85      1774
     neutral       0.54      0.67      0.60      1233
         sad       0.53      0.51      0.52      1247
    surprise       0.71      0.84      0.77       831

    accuracy                           0.65      7178
   macro avg       0.66      0.63      0.64      7178
weighted avg       0.66      0.65      0.65      7178



In [ ]:
torch.save(resnet_model.state_dict(), "resnet_model.pt")

In [ ]:
import os
os.makedirs("emotion_model", exist_ok=True)

In [ ]:
torch.save(cnn_model.state_dict(), "emotion_model/cnn_model.pt")
torch.save(resnet_model.state_dict(), "emotion_model/resnet_model.pt")

import pickle
with open("emotion_model/label_names.pkl", "wb") as f:
    pickle.dump(label_names, f)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * attn

class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channel_attn = ChannelAttention(channels)
        self.spatial_attn = SpatialAttention()

    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x

In [ ]:
resnet_cbam_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
resnet_cbam_model.fc = nn.Linear(resnet_cbam_model.fc.in_features, 7)

resnet_cbam_model.layer1 = nn.Sequential(resnet_cbam_model.layer1, CBAM(64))
resnet_cbam_model.layer2 = nn.Sequential(resnet_cbam_model.layer2, CBAM(128))
resnet_cbam_model.layer3 = nn.Sequential(resnet_cbam_model.layer3, CBAM(256))
resnet_cbam_model.layer4 = nn.Sequential(resnet_cbam_model.layer4, CBAM(512))

resnet_cbam_model = resnet_cbam_model.to(device)
print(resnet_cbam_model.layer1)

Sequential(
  (0): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (1): CBAM(
    (channel_attn): ChannelAttention(
      (avg_pool): AdaptiveAvgPool2d(output_size=1)

In [ ]:
train_loader_cbam = DataLoader(ResNetFERDataset(X_train, y_train), batch_size=64, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss(weight=class_weights_t.to(device))
optimizer = optim.Adam(resnet_cbam_model.parameters(), lr=0.0001)

EPOCHS = 6
for epoch in range(EPOCHS):
    resnet_cbam_model.train()
    total_loss = 0
    for xb, yb in train_loader_cbam:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = resnet_cbam_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss / len(train_loader_cbam))

1 1.2819025310747874
2 0.8483764755566561
3 0.5888462534593315
4 0.35299218985312236
5 0.1672716531944036
6 0.101255676337831


In [ ]:
test_loader_cbam = DataLoader(ResNetFERDataset(X_test, y_test), batch_size=64, shuffle=False)

resnet_cbam_model.eval()
all_preds = []
with torch.no_grad():
    for xb, yb in test_loader_cbam:
        xb = xb.to(device)
        out = resnet_cbam_model(xb)
        preds = torch.argmax(out, dim=1).cpu().numpy()
        all_preds.extend(preds)

resnet_cbam_accuracy = accuracy_score(y_test.numpy(), all_preds)
print(resnet_cbam_accuracy)
print(classification_report(y_test.numpy(), all_preds, target_names=label_names))

0.6518528838116467
              precision    recall  f1-score   support

       angry       0.48      0.69      0.57       958
     disgust       0.69      0.64      0.66       111
        fear       0.52      0.52      0.52      1024
       happy       0.90      0.80      0.85      1774
     neutral       0.64      0.57      0.60      1233
         sad       0.55      0.51      0.53      1247
    surprise       0.78      0.78      0.78       831

    accuracy                           0.65      7178
   macro avg       0.65      0.64      0.64      7178
weighted avg       0.67      0.65      0.66      7178



In [ ]:
torch.save(resnet_cbam_model.state_dict(), "resnet_cbam_model.pt")

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 119.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
import numpy as np
from PIL import Image
from torchvision.models import resnet18

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

@st.cache_resource
def load_models():
    with open("emotion_model/label_names.pkl", "rb") as f:
        classes = pickle.load(f)

    cnn_model = SimpleCNN(num_classes=len(classes))
    cnn_model.load_state_dict(torch.load("emotion_model/cnn_model.pt", map_location="cpu"))
    cnn_model.eval()

    resnet_model = resnet18(weights=None)
    resnet_model.fc = nn.Linear(resnet_model.fc.in_features, len(classes))
    resnet_model.load_state_dict(torch.load("emotion_model/resnet_model.pt", map_location="cpu"))
    resnet_model.eval()

    return cnn_model, resnet_model, classes

cnn_model, resnet_model, classes = load_models()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess(image):
    gray = image.convert("L").resize((48, 48))
    arr = np.array(gray, dtype=np.float32) / 255.0
    tensor48 = torch.tensor(arr).unsqueeze(0).unsqueeze(0)

    tensor224 = F.interpolate(tensor48, size=(224, 224), mode="bilinear", align_corners=False)
    tensor224 = tensor224.repeat(1, 3, 1, 1)
    tensor224 = (tensor224 - mean) / std

    return tensor48, tensor224

st.title("Face Emotion Detector")
st.caption("Comparing custom CNN vs ResNet18 on the same photo")

source = st.radio("Choose input method", ["Upload a photo", "Use webcam"])

image = None
if source == "Upload a photo":
    uploaded = st.file_uploader("Upload a face photo", type=["jpg", "jpeg", "png"])
    if uploaded is not None:
        image = Image.open(uploaded)
else:
    camera = st.camera_input("Take a photo")
    if camera is not None:
        image = Image.open(camera)

if image is not None:
    st.image(image, caption="Input photo", width=200)

    tensor48, tensor224 = preprocess(image)

    with torch.no_grad():
        cnn_probs = torch.softmax(cnn_model(tensor48), dim=1)[0]
        resnet_probs = torch.softmax(resnet_model(tensor224), dim=1)[0]

    cnn_idx = torch.argmax(cnn_probs).item()
    cnn_label = classes[cnn_idx]
    cnn_conf = cnn_probs[cnn_idx].item() * 100

    resnet_idx = torch.argmax(resnet_probs).item()
    resnet_label = classes[resnet_idx]
    resnet_conf = resnet_probs[resnet_idx].item() * 100

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Custom CNN")
        st.write(f"Prediction: **{cnn_label}**")
        st.write(f"Confidence: {cnn_conf:.2f}%")
        st.progress(int(cnn_conf))
    with col2:
        st.subheader("ResNet18")
        st.write(f"Prediction: **{resnet_label}**")
        st.write(f"Confidence: {resnet_conf:.2f}%")
        st.progress(int(resnet_conf))

    st.markdown("---")
    if cnn_label == resnet_label:
        st.success(f"Both models agree: **{resnet_label}**")
    else:
        st.info(f"Models disagree. Final answer: **{resnet_label}** (ResNet18, our best performing model)")

Writing app.py


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!streamlit run app.py &>/content/logs.txt &
!./cloudflared tunnel --url http://localhost:8501

2026-10-01T05:10:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-01T05:10:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-01T05:10:43Z INF +--------------------------------------------------------------------------------------------+
2026-10-01T05:10:43Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-01T05:10:43Z INF |  https://obj-receiver-chicago-led.trycloudflare.com   